In [41]:
import torch
from torch import nn
from d2l import torch as d2l
from torchinfo import summary

In [42]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(nn.BatchNorm2d(in_channels), nn.ReLU(),
                         nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))

In [43]:
class DenseBlock(nn.Module):
    def __init__(self, num_convs, in_channels, out_channels):
        super(DenseBlock, self).__init__()
        layers = []
        for i in range(num_convs):
            layers.append(conv_block(out_channels * i + in_channels, out_channels))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        for blk in self.block:
            Y = blk(x)
            x = torch.cat((x, Y), dim=1)
        return x

In [44]:
blk = DenseBlock(2, 3, 10)

In [45]:
X = torch.randn(4, 3, 8, 8)
Y = blk(X)
Y.shape

torch.Size([4, 23, 8, 8])

In [46]:
# 过渡层
def transition_block(in_channels, out_channels):
    return nn.Sequential(nn.BatchNorm2d(in_channels), nn.ReLU(),
                         nn.Conv2d(in_channels, out_channels, kernel_size=1),  # 指定输出
                         nn.AvgPool2d(kernel_size=2, stride=2))  # 2x2平均池化 输出减半

In [47]:
blk = transition_block(23, 10)
N = blk(Y)
N.shape

torch.Size([4, 10, 4, 4])

In [ ]:
combined_blk = nn.Sequential(
    DenseBlock(2, 3, 10),
    transition_block(23, 10)
)

# 2. 输入尺寸是原始输入尺寸：(4, 3, 8, 8)
summary(combined_blk, input_size=(4, 3, 8, 8))

In [37]:
b1 = nn.Sequential(
    nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm2d(64), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
)  # 第一层

In [38]:
num_channels, growth_rate = 64, 32  # 卷积层的通道数控制增长率

In [39]:
num_convs_in_dense_blocks = [4, 4, 4, 4]
blks = []
for i, num_convs in enumerate(num_convs_in_dense_blocks):
    blks.append(DenseBlock(num_convs, num_channels, growth_rate))
    num_channels += num_convs * growth_rate
    if i != len(num_convs_in_dense_blocks) - 1:
        blks.append(transition_block(num_channels, num_channels//2))  # 添加过渡层
        num_channels = num_channels//2

net = nn.Sequential(b1,*blks,
                    nn.BatchNorm2d(num_channels),nn.ReLU(),
                    nn.AdaptiveAvgPool2d((1,1)),  # 自适应平均池化
                    nn.Flatten(),
                    nn.Linear(num_channels,10)
                    )

In [40]:
summary(net, input_size=(8,1,224,224))

RuntimeError: Failed to run torchinfo. See above stack traces for more details. Executed layers up to: [Sequential: 1, Conv2d: 2, BatchNorm2d: 2, ReLU: 2, MaxPool2d: 2, DenseBlock: 1, Sequential: 3, BatchNorm2d: 4, ReLU: 4, Conv2d: 4, Sequential: 3, BatchNorm2d: 4, ReLU: 4, Conv2d: 4, Sequential: 3, BatchNorm2d: 4, ReLU: 4, Conv2d: 4, Sequential: 3, BatchNorm2d: 4, ReLU: 4, Conv2d: 4, Sequential: 1, BatchNorm2d: 2, ReLU: 2, Conv2d: 2, AvgPool2d: 2]

In [ ]:
lr,num_epochs,batch_size = 0.1,10,256
train_iter,test_iter = d2l.load_data_fashion_mnist(batch_size, resize=96)
d2l.train_ch6(net,train_iter,test_iter,num_epochs,lr,d2l.try_gpu())